In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

print("Generating 10-Year Master Dataset with CLIMATE SHOCKS...")

np.random.seed(42)
dates = pd.date_range(start="2010-01-01", end="2020-12-31", freq='W')
districts = [f"District_{i}" for i in range(1, 53)]

data = []
for d in districts:
    for date in dates:
        month = date.month
        
        # --- THE FIX: THE DISASTER GENERATOR ---
        # 5% chance of a massive compound shock during critical crop windows!
        is_shock = np.random.choice([True, False], p=[0.05, 0.95])
        
        if month in [8, 9] and is_shock: 
            # August/Sept Disaster (Kharif Failure)
            tmax = np.random.normal(43, 2)  # Extreme Heat
            ssm = np.random.normal(0.05, 0.01) # Extreme Drought
        elif month in [1, 2] and is_shock:
            # Jan/Feb Disaster (Rabi Failure)
            tmax = np.random.normal(35, 2)  # Unseasonal Winter Heat
            ssm = np.random.normal(0.05, 0.01)
        else:
            # Normal Weather
            if month in [4, 5, 6]: # Summer
                tmax = np.random.normal(40, 3)
                ssm = np.random.normal(0.08, 0.02)
            elif month in [7, 8, 9]: # Normal Monsoon
                tmax = np.random.normal(32, 2)
                ssm = np.random.normal(0.35, 0.1)
            else: # Normal Winter
                tmax = np.random.normal(25, 4)
                ssm = np.random.normal(0.20, 0.05)
            
        data.append({'Date': date, 'District': d, 'Month': month, 'LST': tmax, 'SSM': ssm})

df = pd.DataFrame(data)

# 2. CROP PHENOLOGY
def get_crop_window(month):
    if month in [8, 9]: return 'Kharif_Critical'
    if month in [1, 2]: return 'Rabi_Critical'
    return 'Non_Critical'

df['Crop_Window'] = df['Month'].apply(get_crop_window)

# 3. THRESHOLDS (Simplified for the API alignment)
# We define anomalies strictly so the API matches the training data
df['Heat_Anomaly'] = (df['LST'] >= 42.0).astype(int)
df['Drought_Anomaly'] = (df['SSM'] <= 0.08).astype(int)

# 4. COMPOUND STRESS LOGIC
df['Compound_Stress'] = ((df['Heat_Anomaly'] == 1) & 
                         (df['Drought_Anomaly'] == 1) & 
                         (df['Crop_Window'] != 'Non_Critical')).astype(int)

# 5. TARGET VARIABLE (95% chance of crop death if compound stress hits)
df['Target_Failure'] = np.where(
    df['Compound_Stress'] == 1,
    np.random.choice([1, 0], p=[0.95, 0.05], size=len(df)),
    np.random.choice([1, 0], p=[0.01, 0.99], size=len(df)) # 1% baseline noise
)

print(f"Dataset generated: {len(df)} rows.")
print(f"Total Crop Failures to learn from: {df['Target_Failure'].sum()}")

# 6. TRAIN MODEL
df = pd.get_dummies(df, columns=['Crop_Window'], drop_first=False)

features = ['LST', 'SSM', 'Heat_Anomaly', 'Drought_Anomaly', 
            'Crop_Window_Kharif_Critical', 'Crop_Window_Rabi_Critical', 'Crop_Window_Non_Critical']

# Ensure all columns are float/int for XGBoost
X = df[features].astype(float)
y = df['Target_Failure']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = xgb.XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, scale_pos_weight=2)
model.fit(X_train, y_train)

print("\n--- MASTER MODEL PERFORMANCE ---")
preds = model.predict(X_test)
print(classification_report(y_test, preds))

# Save the super-accurate model
model.save_model("../src/csi_model_v2.json")
print("✅ Saved as csi_model_v2.json") 

Generating 10-Year Master Dataset with CLIMATE SHOCKS...
Dataset generated: 29848 rows.
Total Crop Failures to learn from: 434

--- MASTER MODEL PERFORMANCE ---
              precision    recall  f1-score   support

           0       0.99      1.00      1.00      5880
           1       1.00      0.36      0.52        90

    accuracy                           0.99      5970
   macro avg       1.00      0.68      0.76      5970
weighted avg       0.99      0.99      0.99      5970

✅ Saved as csi_model_v2.json
